In [15]:
# -----------------------------
# 03_train_efficientnet.ipynb
# -----------------------------

# Importations
import numpy as np
import torch
import os
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split
from torchvision import models, transforms
import torch.optim as optim
import mlflow
import mlflow.pytorch
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights


In [16]:
# -----------------------------
# Paramètres
# -----------------------------
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "efficientnet_model.pth"

IMAGE_SIZE = 224

In [17]:

# -----------------------------
# Chargement des données
# -----------------------------

# Si grayscale → 1 canal
# Transforme en tenseur PyTorch et ajoute dimension canal
PROCESSED_DIR = "../data/processed"

# Charger les fichiers numpy
images = np.load(os.path.join(PROCESSED_DIR, "images.npy"))  # shape = [N,224,224,1]
labels = np.load(os.path.join(PROCESSED_DIR, "labels.npy"))
X = torch.tensor(images, dtype=torch.float32).unsqueeze(1)  # shape = [N,1,224,224]
y = torch.tensor(labels, dtype=torch.long)

# Créer Dataset et DataLoader
dataset = TensorDataset(X, y)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train batch shape:", next(iter(train_loader))[0].shape)
print("Validation batch shape:", next(iter(val_loader))[0].shape)



Train batch shape: torch.Size([32, 1, 224, 224, 1])
Validation batch shape: torch.Size([32, 1, 224, 224, 1])


In [18]:
# -----------------------------
# Définition du modèle EfficientNet (B0)
# -----------------------------

weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

# Adapter la première couche pour 1 canal
old_conv = model.features[0][0]
new_conv = nn.Conv2d(
    in_channels=1,
    out_channels=old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
    bias=False
)
with torch.no_grad():
    new_conv.weight[:, 0, :, :] = old_conv.weight.mean(dim=1)
model.features[0][0] = new_conv

# Modifier la dernière couche pour 2 classes
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
model = model.to(DEVICE)

In [19]:

# Loss et optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [20]:
# -----------------------------
# Configuration MLflow
# -----------------------------
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("EfficientNet_Model")


2025/12/16 12:50:46 INFO mlflow.tracking.fluent: Experiment with name 'EfficientNet_Model' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///c:/Users/USER/Desktop/ecg-classification/notebooks/mlruns/574370398224437146', creation_time=1765885846028, experiment_id='574370398224437146', last_update_time=1765885846028, lifecycle_stage='active', name='EfficientNet_Model', tags={}>

In [26]:
# -----------------------------
# Réinitialisation des runs fantômes
# -----------------------------
# Vérifie si un run fantôme existe

if mlflow.active_run() is not None:
    print("Run actif trouvé :", mlflow.active_run().info.run_id)
    mlflow.end_run()
    print("Run actif terminé ✅")

Run actif trouvé : 39ec39d433ad404d90486b32065d9db5
Run actif terminé ✅


In [27]:
# -----------------------------
# Entraînement avec MLflow
# -----------------------------
with mlflow.start_run(run_name="EfficientNet_Run") as run:

    # Log des paramètres
    mlflow.log_param("model", "EfficientNet_B0")
    mlflow.log_param("epochs", EPOCHS)
    mlflow.log_param("device", str(DEVICE))

    for epoch in range(EPOCHS):
        train_loss = 0.0
        train_acc = 0.0
        val_loss = 0.0
        val_acc = 0.0
        
        # ======= Boucle d'entraînement =======
        model.train()
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            
            # Supprimer la dimension en trop si nécessaire
            if inputs.dim() == 5 and inputs.size(-1) == 1:
                inputs = inputs.squeeze(-1)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total = targets.size(0)
            train_acc += (predicted == targets).sum().item()

        train_loss /= len(train_loader.dataset)
        train_acc /= len(train_loader.dataset)

        # ======= Boucle de validation =======
        model.eval()
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                if inputs.dim() == 5 and inputs.size(-1) == 1:
                    inputs = inputs.squeeze(-1)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                val_acc += (predicted == targets).sum().item()

        val_loss /= len(val_loader.dataset)
        val_acc /= len(val_loader.dataset)

        # Log metrics dans MLflow
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("train_acc", train_acc, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        mlflow.log_metric("val_acc", val_acc, step=epoch)

    # ======= Sauvegarde du modèle =======
    model.eval()
    mlflow.pytorch.log_model(
        pytorch_model=model,
        artifact_path="efficientnet_model",
    )

    print("Run MLflow terminé avec succès :", run.info.run_id)

c:\Users\USER\anaconda3\envs\ecg_project\lib\site-packages\_distutils_hack\__init__.py:15: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
c:\Users\USER\anaconda3\envs\ecg_project\lib\site-packages\_distutils_hack\__init__.py:30: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


Run MLflow terminé avec succès : ce199bb39f4b492497f26c6525755c6d
